In [0]:
from pyspark.sql.functions import *

In [0]:
website=spark.read.format("json")\
    .option("multiline","true")\
    .option("inferSchema","true")\
    .option("mode","PERMISSIVE")\
    .load("s3://retail-lakehouse-ashu/raw/website/")

In [0]:
if "_corrupt_record" in website.columns:
    website.cache()
    website.count()

    corrupt_count=website.filter(col("_corrupt_record").isNotNull()).count()
    print(f"Corrupt Records: {corrupt_count}")
else:
    print("No _corrupt_record column present. JSON parsed successfully.")

In [0]:
website.printSchema()

In [0]:
website.show(5)

In [0]:
validation_rule=(
col("customer_id").isNotNull() &
(size(col("items")) > 0) &
col("order_id").isNotNull() &
col("order_status").isNotNull() &
col("order_timestamp").isNotNull() &
col("shipping_address.city").isNotNull() &
col("shipping_address.pincode").isNotNull() &
col("shipping_address.city").isNotNull() &
col("shipping_address.state").isNotNull() &
col("payment_amount").isNotNull()
)

In [0]:
good_df=website.filter(validation_rule)
good_df.display()

In [0]:
bad_df=website.filter(~validation_rule)
bad_df.display()

In [0]:
bronze_website=good_df.withColumn("ingestion_timestamp",current_timestamp())\
    .withColumn("source_system",lit("website"))\
    .withColumn("source_file",col("_metadata.file_path"))\
    .withColumn("batch_id",lit(1))

In [0]:
bronze_website.display()

In [0]:
total_records=website.count()

good_records=good_df.count()

bad_records=bad_df.count()

print(f"Total Records   : {total_records}")
print(f"Good Records    : {good_records}")
print(f"Bad Records     : {bad_records}")

In [0]:
duplicate_df=website.groupBy(col("order_id")).count().filter(col("count") > 1)

In [0]:
assert total_records==good_records+bad_records

In [0]:
bronze_website.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3://retail-lakehouse-ashu/bronze/website/")

In [0]:
bad_df=(
    bad_df
        .withColumn("batch_id", lit(1))
        .withColumn("pipeline_name", lit("bronze_website"))
        .withColumn("source_system", lit("website"))
        .withColumn("table_name", lit("website"))
        .withColumn("failure_reason", lit("MANDATORY_FIELD_VALIDATION_FAILED"))
        .withColumn("source_file", col("_metadata.file_path"))   # replace later with metadata
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("raw_record", to_json(struct(*bad_df.columns)))
        .select(
        col("batch_id"),
        col("pipeline_name"),
        col("source_system"),
        col("table_name"),
        col("failure_reason"),
        col("source_file"),
        col("ingestion_timestamp"),
        col("raw_record")
    )
)

In [0]:
bad_df.write \
    .format("delta") \
    .mode("append") \
    .save("s3://retail-lakehouse-ashu/bad_records/")